# apc — startup: home the rail

Bench startup for the apc cell: bring the **rail** (the robot's linear axis) to a known reference before any motion. Run this once after powering on, before the workflow / calibration.

It does the standard 3-step axis init — `set_axis` (encoder/microstep) → `set_pid` → `home_with_stop` — exactly like the old project startup notebooks, but reads every value from `core.rail_cfg` (which deep-merges the core defaults). The home offset is `core.rail_cfg["offset"]` — the new home of `core.rail_offset`, which no longer exists.

> ⚠️ `home_with_stop(dir=-1)` drives the rail into its hard stop. Make sure the carriage path is clear and the e-stop is in reach before running the rail cell.

In [ ]:
import os, sys, importlib, pkgutil, time
from pathlib import Path

# apc project folder. Not strictly needed for homing (nothing is written), but
# keeps scene paths + component imports resolving like main.py.
_HERE = Path("/home/dorna/Downloads/projects/apc")
os.chdir(_HERE)
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

# Register component types before the scene loads (library + project-local).
import workspace.components  # noqa: F401
comp_dir = _HERE / "components"
if comp_dir.is_dir():
    for m in pkgutil.iter_modules([str(comp_dir)]):
        if not m.name.startswith("_"):
            importlib.import_module(f"components.{m.name}")

from workspace.workspace import Workspace

In [ ]:
# Production scene — homing the rail is tool-independent, so use the real
# bench layout (the suction gripper is mounted; that's fine).
SCENE = ["scene/core_500.j2", "scene/layout.j2"]
ws = Workspace(config_path=[str(_HERE / p) for p in SCENE], port=5000)
core = ws.components["core"]
print("has_rail:", core.has_rail, "| rail type:", core.rail_cfg["type"])
print("rail axis:", core.rail_cfg["axis"], "| home offset:", core.rail_cfg["offset"])
print("rail travel limits (mm):", core.rail_min, "->", core.rail_max)

## Real hardware

Homing only does something on the real controller — in simulation the SDK calls are stubs that return immediately. Turn simulation **off** to home for real.

In [ ]:
core.simulation(False)
assert not core._simulation_mode, "core still in simulation — homing would be a no-op"

## Home the rail

3-step init from `core.rail_cfg`, with a 1 s settle between SDK calls. `dir=-1` homes toward the negative hard stop; the rail position is set to `rail_cfg["offset"]` once the stop is found.

Equivalent one-liner if you have a recipe handy: `recipe.set_axis_with_stop(core.rail_cfg, dir=-1)` (bundles these three calls, pause-aware via `rt.delay`).

In [ ]:
cfg = core.rail_cfg
api = core.robot_api

# 1) axis: encoder + microstep config
api.set_axis(
    index=cfg["axis"],
    usem=cfg["usem"], usee=cfg["usee"],
    pprm=cfg["pprm"], tprm=cfg["tprm"],
    ppre=cfg["ppre"], tpre=cfg["tpre"],
)
time.sleep(1)

# 2) pid: closed-loop gains
api.set_pid(
    index=cfg["axis"],
    p=cfg["p"], i=cfg["i"], d=cfg["d"],
    duration=cfg["duration"], threshold=cfg["threshold"],
)
time.sleep(1)

# 3) home against the hard stop; set the rail to rail_cfg['offset'] there
api.home_with_stop(index=cfg["axis"], val=cfg["offset"], dir=-1)
print("rail homing commanded.")

## Verify

Read the live joints back; the rail sits at index `rail_cfg["axis"]` and should read the home offset right after homing.

In [ ]:
core._last_joints = None
core.update_pose()
j = list(core.robot_api.joint())
ax = core.rail_cfg["axis"]
print("joints:", [round(v, 3) for v in j])
print(f"rail (axis {ax}) position:", round(j[ax], 3), "mm  (home offset =", core.rail_cfg["offset"], ")")